# 00 Prepare split FF++ PNG images on Kaggle
Expected layout: `train/{real,fake}/*.png`, `val/{real,fake}/*.png`,
`test/{real,fake}/*.png`. The setup indexes the existing PNG files directly;
it does not extract videos, detect faces, download split lists, or re-split images.
Images should already be suitable face crops; inspect the preview below.


## Runtime and embedded project setup
For training, select **Runtime → Change runtime type → GPU**. This cell unpacks
the complete project source into `/kaggle/working/freqblender_project` and installs support
dependencies. It keeps Kaggle's matched PyTorch/torchvision CUDA installation.
No GitHub fork or separate project ZIP upload is needed.


## Kaggle input and writable outputs
Attach `delno002/split-ffpp` to every notebook. Its root must contain
train/real, train/fake, val/real, val/fake, test/real and test/fake PNG folders.
Existing splits are preserved. Images are read directly from the attached dataset.
Save notebook outputs before ending a session. For later stages, attach the previous
saved output and set PREVIOUS_RUNS to its `freqblender/runs` directory.


In [ ]:
import shutil
import sys, subprocess
from pathlib import Path



FFPP_ROOT = Path("/kaggle/input/split-ffpp")  # EDIT if Kaggle mounts a different path
DRIVE_PROJECT = Path("/kaggle/working/freqblender")  # legacy variable name, local Kaggle path
DATA_ROOT = DRIVE_PROJECT / "prepared"
RUN_ROOT = DRIVE_PROJECT / "runs"
MANIFEST = DATA_ROOT / "manifest.csv"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PREVIOUS_RUNS = None  # e.g. Path("/kaggle/input/previous-output/freqblender/runs")
if PREVIOUS_RUNS is not None:
    # Copy missing files only, preserving newer checkpoints from this session.
    previous = Path(PREVIOUS_RUNS)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    for source in previous.rglob("*"):
        target = RUN_ROOT / source.relative_to(previous)
        if source.is_file() and not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
from freqblender.data import build_image_manifest
MANIFEST = build_image_manifest(FFPP_ROOT, MANIFEST)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(DRIVE_PROJECT / "environment.txt").write_text(versions)
print("Manifest:", MANIFEST)
print("Using image-level samples; original video/source identities are unavailable.")


## Inspect the manifest
Labels: real=0, fake=1. Each PNG is one sample. `video_id` and `group_id`
contain image identifiers solely for compatibility with the training code.
They are not recovered video identities. Consequently evaluation is image-level,
and source-video leakage cannot be ruled out from this folder layout alone.


In [ ]:
from freqblender.data import read_manifest
table = read_manifest(MANIFEST)
display(table.groupby(["split", "label"]).size().unstack(fill_value=0).rename(columns={0: "real", 1: "fake"}))
display(table.head())


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
preview = table[table.split == "train"].sample(min(12, len(table[table.split == "train"])), random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for axis in axes.flat:
    axis.axis("off")
for axis, (_, row) in zip(axes.flat, preview.iterrows()):
    with Image.open(row.path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"{Path(row.path).name} | label={row.label}", fontsize=8)
plt.tight_layout()


Continue with **01_train_freqblender.ipynb** and attach the same PNG dataset.
Each notebook recreates the manifest without copying images. Save trained `runs/`
outputs between sessions and restore them using PREVIOUS_RUNS in the setup cell.
